In [ ]:
import os
from google.colab import drive

# 1. Mount Google Drive
drive.mount('/content/drive')

# 2. Điền thông tin Kaggle API của bạn vào đây
# (Vào Kaggle -> Settings -> Creatxe New Token để lấy username và key)
os.environ['KAGGLE_USERNAME'] = "huyphngquc"
os.environ['KAGGLE_KEY'] = "d2051dddb134d7809d29527c977cab58"

# Tạo thư mục trên Drive để lưu dataset cố định (nếu chưa có)
DRIVE_DATASET_DIR = '/content/drive/MyDrive/Thesis_ViVQA_Data'
os.makedirs(DRIVE_DATASET_DIR, exist_ok=True)

In [ ]:
import os
import time

LOCAL_DATA_DIR = '/content/vivqa_dataset'
ZIP_PATH_ON_DRIVE = f'{DRIVE_DATASET_DIR}/vivqa-1.zip' # Đảm bảo biến này đúng
ZIP_PATH_LOCAL = '/content/vivqa-1.zip'

start_time = time.time()

# 1. Xóa rác cũ nếu có (Quan trọng để tránh giải nén đè lên file lỗi)
!rm -rf {LOCAL_DATA_DIR}
!rm -f {ZIP_PATH_LOCAL}

# 2. Copy bằng lệnh Linux (Trâu hơn shutil)
print("Đang copy từ Drive sang Local...")
!cp "{ZIP_PATH_ON_DRIVE}" "{ZIP_PATH_LOCAL}"

# 3. Giải nén và TỰ ĐỘNG SỬA LỖI (Dùng option -FF nếu cần)
print("Đang giải nén...")
os.makedirs(LOCAL_DATA_DIR, exist_ok=True)

# Thử giải nén bình thường trước
result = !unzip -q -n {ZIP_PATH_LOCAL} -d {LOCAL_DATA_DIR}

# Kiểm tra nếu vẫn còn báo lỗi "bad zipfile offset"
if any("bad zipfile offset" in line for line in result):
    print("⚠️ File zip có vẻ bị lỗi offset, đang thử fix...")
    # Lệnh fix file zip của linux
    !zip -F {ZIP_PATH_LOCAL} --out {ZIP_PATH_LOCAL}_fixed.zip
    !unzip -q -n {ZIP_PATH_LOCAL}_fixed.zip -d {LOCAL_DATA_DIR}

# 4. Dọn dẹp
if os.path.exists(ZIP_PATH_LOCAL): os.remove(ZIP_PATH_LOCAL)

print(f"✅ Xong! Kiểm tra thử: {len(os.listdir(LOCAL_DATA_DIR))} folders/files.")

In [ ]:
# 4. Kiểm tra và in ra cấu trúc thực tế
print("\n--- KIỂM TRA CẤU TRÚC DATA ---")
if os.path.exists(LOCAL_DATA_DIR):
    # Liệt kê tất cả file và folder (kể cả trong folder con)
    all_files = []
    for root, dirs, files in os.walk(LOCAL_DATA_DIR):
        for name in files:
            all_files.append(os.path.join(root, name))
    
    print(f"📍 Path cơ sở: {LOCAL_DATA_DIR}")
    print(f"📦 Tổng số lượng file tìm thấy: {len(all_files)}")
    
    if len(all_files) > 0:
        print("📄 5 file đầu tiên để check path:")
        for f in all_files[:5]:
            print(f"   -> {f}")
    else:
        print("❌ CẢNH BÁO: Thư mục tồn tại nhưng không có file nào bên trong (do giải nén lỗi).")
else:
    print("❌ LỖI: Thư mục LOCAL_DATA_DIR không tồn tại.")

In [ ]:
!pip install -q transformers timm accelerate bitsandbytes scikit-learn underthesea

In [ ]:
import warnings, logging
warnings.filterwarnings("ignore", category=UserWarning)
logging.getLogger("torch.utils.data.dataloader").setLevel(logging.ERROR)

In [ ]:
import torch

# ==========================================
# BOX CONFIG — Chỉnh tham số tại đây
# ==========================================
BOX_CONFIG = {
    "train_csv": "/content/vivqa_dataset/ViVQA-main/ViVQA-main/train.csv",
    "train_img_dir": "/content/vivqa_dataset/train",
    "train_box_csv": "/content/vivqa_dataset/merged_train.csv",

    "test_csv": "/content/vivqa_dataset/ViVQA-main/ViVQA-main/test.csv",
    "test_img_dir": "/content/vivqa_dataset/test",

    "save_path": "/content/vivqa_dataset/vivqa_box.pth",

    "blip_model": "Salesforce/blip2-opt-2.7b",
    "text_model": "vinai/phobert-base",

    "batch_size": 32,
    "epochs": 25,
    "lr": 5e-5,
    "weight_decay": 0.05,

    "lambda_bbox": 2.0,

    "type_embed_dim": 64,
    "device": "cuda" if torch.cuda.is_available() else "cpu",

    "max_length": 50,
    "patience": 5,
    "delta": 0.001,
    "num_workers": 2,

    # Trọng số loss theo q_type (tăng = ép model học kỹ hơn loại đó)
    "type_loss_weights": {0: 4.0, 1: 0.5, 2: 4.0, 3: 4.0},
}


In [ ]:
# ==========================================
# V4 CONFIG — Chỉnh tham số tại đây
# ==========================================
V4_CONFIG = {
    # File train gốc (tự tách 90/10 thành Train + Val nội bộ)
    "train_csv": "/content/vivqa_dataset/ViVQA-main/ViVQA-main/train.csv",
    "train_img_dir": "/content/vivqa_dataset/train",

    # File test (hold-out, chỉ dùng cho inference)
    "test_csv": "/content/vivqa_dataset/ViVQA-main/ViVQA-main/test.csv",
    "test_img_dir": "/content/vivqa_dataset/test",

    # File mapping loại câu hỏi → đáp án (tạo Hierarchical Mask)
    "type_mapping_csv": "/content/vivqa_dataset/answer_type_mapping.csv",

    "save_path": "/content/vivqa_dataset/vivqa_v4.pth",

    "blip_model": "Salesforce/blip2-opt-2.7b",
    "text_model": "vinai/phobert-base",

    "batch_size": 32,
    "epochs": 25,
    "lr": 5e-5,
    "weight_decay": 0.05,
    "lambda_type": 0.5,

    "device": "cuda" if torch.cuda.is_available() else "cpu",

    "max_length": 50,
    "patience": 5,
    "num_workers": 2,

    # Trọng số loss theo q_type (boost type 1 vì khó nhất)
    "type_loss_weights": {0: 1.0, 1: 3.0, 2: 1.0, 3: 1.0},
}


# ViVQA — Main Training Notebook

Notebook này điều phối quá trình huấn luyện và inference cho **hai mô hình**:

| Mô hình | Thư mục | Mô tả |
|---------|---------|-------|
| **Box** | `Box/` | Đa nhiệm: phân loại câu trả lời + dự đoán Bounding Box |
| **V4**  | `V4/`  | Hierarchical VQA với Type Mask theo loại câu hỏi |

**Cấu trúc thư mục:**
```
DHM/
├── main.ipynb          ← Notebook điều phối (file này)
├── utils/
│   ├── preprocessing.py   ← preprocess_vietnamese, find_image_path
│   ├── early_stopping.py  ← EarlyStopping
│   └── fusion.py          ← CrossAttentionFusion
├── Box/
│   ├── dataset.py         ← ViVQADataset, ViVQAMultiTaskInferenceDataset
│   ├── model.py           ← PhoBERT_BLIP2_EfficientNet_MultiTask
│   ├── trainer.py         ← Vòng lặp huấn luyện
│   └── inference.py       ← Chạy inference và xuất CSV
└── V4/
    ├── dataset.py         ← ViVQADataset, ViVQAInferenceDataset
    ├── model.py           ← PhoBERT_BLIP2_VQA_Hierarchical, create_answer_type_mask
    ├── trainer.py         ← Vòng lặp huấn luyện
    └── inference.py       ← Chạy inference và xuất CSV
```

In [ ]:
import sys
import os

# Đảm bảo thư mục DHM/ nằm trong Python path để import được các module
notebook_dir = os.path.dirname(os.path.abspath("main.ipynb"))
if notebook_dir not in sys.path:
    sys.path.insert(0, notebook_dir)

## Mô hình 1 — Box (Đa nhiệm: VQA + Bounding Box)

**Mô tả:** Dùng BLIP-2 (Q-Former) + EfficientNet-B7 + PhoBERT kết hợp qua CrossAttention.  
Hai đầu ra: phân loại câu trả lời và hồi quy tọa độ Bounding Box (chuẩn hóa [0, 1]).  
Có thể chỉnh `CONFIG` trong `Box/config.py`.

In [ ]:
from Box.trainer import train as train_box

train_box(BOX_CONFIG)

In [ ]:
from Box.inference import run_inference_and_save as box_inference

box_inference(BOX_CONFIG)

## Mô hình 2 — V4 (Hierarchical VQA với Type Mask)

**Mô tả:** Dùng BLIP-2 Vision Encoder (raw features, dim=1408) + PhoBERT kết hợp qua CrossAttention.  
Hai đầu ra: dự đoán loại câu hỏi (type classifier) + phân loại câu trả lời với mask theo từng loại câu hỏi.  
Tập train được tự chia thành Train/Val (90/10) theo stratify `type`.  
Có thể chỉnh `CONFIG` trong `V4/config.py`.

In [ ]:
from V4.trainer import train as train_v4

train_v4(V4_CONFIG)

In [ ]:
from V4.inference import run_inference_and_save as v4_inference

v4_inference(V4_CONFIG)